# 第 06 章练习

## 6.7 章节实践

完成基础题、手算和两种算子的运行检查，再从 SpMV、RoPE 的参数比较中任选一项。下面提供了运行单元格；参数修改和结果分析由你完成。


先完成 SpMV 和 RoPE 实验，再使用本次结果作答。本 Notebook 始终从第 06 章目录运行，各单元格已写好子实验路径。


<a id="s-06-02_chapter_test"></a>


### 一、基础理解

1. CSR 的 row_ptr、col_idx、values 分别表示什么？
2. RoPE 的 12 个 head、每个 64 通道，为什么能拆成 384 个旋转对？
3. FP32 转 BF16 后，为什么完整 CSR 的存储量不会减半？
4. RoPE 在 position=0 时应输出什么？
5. 同一 Stream 上，提交 kernel 后为什么还要同步再校验？
6. SIMD 示例使用哪类 Core？SIMD 与 SIMT 分别怎样分配计算任务？
7. 仅有 Ascend 950 源码模板，能否说明 SIMT 实验已经通过？

[综合题参考答案](answer/06.07_chapter_test_answer.md)；[RoPE 参考答案](02_rope_operator/answer/06.07_rope_answer.md)


### 二、手算数据流

手算两行 CSR SpMV 和一个 RoPE 旋转对。把手算输入对应到源码中的设备缓冲区，说明 Host 如何准备输入、Device 如何计算、结果如何回读。


### 三、检查两种算子的结果

可使用前面实验已生成的日志和 CSV；需要重测时，运行下面相应部分的代码。保存设备信息、误差和时间，说明计时是否包含传输、初始化和同步。


### 四、选择一个问题继续实验

任选一项：

- 在 SpMV Host 原型中改变分区候选，固定矩阵和重复次数，比较初始化、稳态时间与误差。
- 在 RoPE SIMD 源码中调整一个缓冲区参数，保持输入布局和 4 个 position case，比较修改前后。

先验证数值，再解释性能变化。A3 上运行 SIMD 路径；950 SIMT 文件用于阅读线程映射，实测需要对应硬件。


<a id="s-01-06_chapter_test"></a>


## SpMV：读取结果并比较参数

复用上一实验结果时，跳过下面的构建运行格，在读取格中把 `chapter_test.csv` 改为 `course_quick_check.csv`。本节重新运行后，使用默认的 `chapter_test.csv`。

参数比较在 CPU 原型中进行，只改变分区候选数或精度，保持矩阵、预热和重复次数相同。


### 检查设备和 CMake

下面显示 CANN 目录、CMake 版本和 NPU 信息。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info


In [ ]:
%%bash
set -e
cd 01_ascendc_spmv/src/ascend_spmv
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j
# 学生修改：--matrix 选择输入矩阵；--csv 指定本次结果文件。
./build/bin/spmv_benchmark --matrix U1 --warmup 1 --repeat 3 --csv results/chapter_test.csv


In [ ]:
import csv
from pathlib import Path

with Path("01_ascendc_spmv/src/ascend_spmv/results/chapter_test.csv").open() as stream:
    row = next(csv.DictReader(stream))
print("实现、位置 | 初始化 ms | 每次总时间 ms | 相对误差")
print("FP32、NPU |", row["ascendc_init_ms"], row["ascendc_total_ms"], row["ascendc_error"])
print("BF16、CPU |", row["host_prototype_bf16_init_ms"],
      row["host_prototype_bf16_total_ms"], row["host_prototype_bf16_mixed_error"])
print("BF16 persistent、CPU/OpenMP |", row["host_prototype_bf16_persistent_init_ms"],
      row["host_prototype_bf16_persistent_total_ms"], row["host_prototype_bf16_persistent_error"])


### 练习

#### 1. 基础题

（1）运行 Ascend C 版本后，CSV 的 `actual_backend` 应是什么？

（2）在 `include/ascendc_spmv.hpp` 中找到设备后端的类名。

（3）`ascendc_launch_to_complete_ms` 包含哪些工作？

#### 2. 读取结果

保存一组运行命令与 CSV，解释初始化、每次计算总时间、压缩比、分区均衡度和误差，标出各实现的计算位置。

#### 3. 单变量比较

在 CPU 原型中选择分区候选数或精度作为变量，固定矩阵、预热和重复次数。比较修改前后的初始化、计算时间与误差。

学生修改位置：[npu_spmv_context.cpp](01_ascendc_spmv/src/ascend_spmv/src/npu_spmv_context.cpp) 中 `HostPrototypeBf16PersistentBackend::prepare()` 的 `candidates` 列表。先记录原来的候选集合，再只改变一个候选值，重新构建并保存到另一份 CSV。

#### 4. 追踪执行过程

从 `prepare()` 和 `run()` 追踪设备内存分配、传输、RTC 编译与启动。结合运行日志解释 FP32 设备实现与 CPU 原型的不同。

[SpMV 参考答案](01_ascendc_spmv/answer/06.07_spmv_answer.md)


<a id="s-02-03_chapter_test"></a>


下面运行 RoPE。修改 SIMD 参数后再运行一次，并换一个日志文件名保存结果。


## RoPE：运行检查与参数比较

先用 CPU reference 检查 position=0，再检查 SIMD 程序的 4 组输入。扩展题只调整一个参数。


### CPU reference

使用两个旋转 pair 构造最小 reference，并验证 position=0 时输出与输入完全相同。


In [ ]:
import numpy as np

x_even = np.array([0.25, -0.50], dtype=np.float32)
x_odd = np.array([0.75, 0.125], dtype=np.float32)
cos = np.ones_like(x_even)
sin = np.zeros_like(x_even)
out_even = x_even * cos - x_odd * sin
out_odd = x_even * sin + x_odd * cos
np.testing.assert_array_equal(out_even, x_even)
np.testing.assert_array_equal(out_odd, x_odd)
print("POSITION_ZERO_REFERENCE=PASS")


### 运行 SIMD

完成[上一节](02_rope_operator/06_rope.ipynb#s-02-02_rope_simd_simt)的 Host 编译后，使用 2 次预热、10 次重复运行当前源码，查看 `status=PASS`、`cases=4/4` 和最大误差。


In [ ]:
%%bash
set -euo pipefail
mkdir -p results
# 学生修改：首次保留 baseline；修改 Kernel 后改为 results/rope_candidate.log。
LOG=results/rope_baseline.log
./02_rope_operator/build/rope_simd_rtc \
  --kernel 02_rope_operator/src/rope_simd_kernel.cpp --warmup 2 --repeat 10 | tee "$LOG"


### SIMD 参数实验

学生修改位置：[rope_simd_kernel.cpp](02_rope_operator/src/rope_simd_kernel.cpp) 开头的 `BUFFER_NUM`。比较 1 与 2，保持旋转公式、输入、4 组 position、预热和重复次数相同。

修改后重新运行上面的程序，它会读取并编译当前 Kernel。两次都通过数值校验后，比较包含提交与同步的 `device_mean_us`。

当前程序只有一个 tile，依次 CopyIn→Compute→CopyOut。把 BUFFER_NUM 改为 2 不会自动形成多 tile 流水；本题观察是否有明显变化，并解释原因。


In [ ]:
!grep ROPE_RESULT results/rope_baseline.log
# 完成参数比较后，取消下一行注释查看另一组结果。
# !grep ROPE_RESULT results/rope_candidate.log


完成本节练习后查看：[参考答案](02_rope_operator/answer/06.07_rope_answer.md)
